# Fetch model → upload to S3/MinIO

Requires workbench **env** from `minio` and `huggingface-token` (same keys as `fetch_model.py`). Optional: `MODEL_ID`, `S3_FOLDER`.

In [ ]:
%pip install -q boto3 huggingface_hub
# Check GPU
!nvidia-smi
# Check env variables
#!env | grep -E 'HF_TOKEN|AWS'

import os
for k in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_S3_BUCKET", "HF_TOKEN"]:
    v = os.environ.get(k, "")
    ok = bool(v) and v not in ("<YOUR_HF_TOKEN>",)
    show = f"{v[:4]}…" if (v and ("SECRET" in k or k == "HF_TOKEN")) else (v or "(missing)")
    print(f"{'OK' if ok else '!!'}  {k}: {show}")

%pip install -q requests

# Check HF token
import os
import requests

t = os.environ.get("HF_TOKEN", "").strip()
if t and t != "<YOUR_HF_TOKEN>":
    r = requests.get("https://huggingface.co/api/whoami-v2", headers={"Authorization": f"Bearer {t}"}, timeout=30)
    print("HF whoami:", r.status_code, r.json() if r.ok else r.text[:200])
else:
    print("Set HF_TOKEN (huggingface-token secret) to test the hub.")

In [ ]:
import os
import boto3
import urllib3
from huggingface_hub import snapshot_download

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def _env(name: str) -> str:
    v = os.environ.get(name, "").strip()
    if not v:
        raise SystemExit(
            f"Missing {name}. Add minio + huggingface-token data connections (envFrom) in the workbench."
        )
    return v


hf_token = _env("HF_TOKEN")
if hf_token in ("<YOUR_HF_TOKEN>", "changeme"):
    raise SystemExit("Set HF_TOKEN (huggingface-token secret).")

aws_key = _env("AWS_ACCESS_KEY_ID")
aws_secret = _env("AWS_SECRET_ACCESS_KEY")
endpoint = _env("AWS_S3_ENDPOINT")
bucket_name = (os.environ.get("AWS_S3_BUCKET") or "models").strip()
verify_ssl = os.environ.get("S3_VERIFY_SSL", "false").lower() in ("1", "true", "yes")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

model_id = os.environ.get("MODEL_ID", "openai/gpt-oss-20b")
s3_folder = os.environ.get("S3_FOLDER", "gpt-oss-20b")

print(f"--- Downloading {model_id} ---")
local_path = snapshot_download(
    repo_id=model_id,
    token=hf_token,
    allow_patterns=["*.json", "*.bin", "*.safetensors", "*.py", "*.txt"],
    ignore_patterns=["*.msgpack", "*.h5"],
)

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=aws_key,
    aws_secret_access_key=aws_secret,
    verify=verify_ssl,
)
try:
    s3.create_bucket(Bucket=bucket_name)
    print(f"Created bucket: {bucket_name}")
except Exception:
    pass

print(f"--- Uploading to {bucket_name}/{s3_folder} ---")
for root, _, files in os.walk(local_path):
    for file in files:
        full_path = os.path.join(root, file)
        rel_path = os.path.relpath(full_path, local_path)
        s3_key = f"{s3_folder}/{rel_path}"
        print(f"Uploading {rel_path}...")
        s3.upload_file(full_path, bucket_name, s3_key)

print("\nDONE:", f"s3://{bucket_name}/{s3_folder}/")